# E-Commerce Data Exploratory Analysis

This notebook provides an exploratory analysis of the generated e-commerce data using both Pandas and PySpark.

## Setup and Data Loading

In [ ]:
# Import required libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession
import sys
sys.path.append('../src')

# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Create Spark session
spark = SparkSession.builder \
    .appName("Exploratory Analysis") \
    .master("local[*]") \
    .getOrCreate()

In [ ]:
# Load data files
customers_df = pd.read_csv('../data/raw/customers.csv')
products_df = pd.read_csv('../data/raw/products.csv')
orders_df = pd.read_csv('../data/raw/orders.csv')
order_items_df = pd.read_csv('../data/raw/order_items.csv')

# Convert to Spark DataFrames
spark_customers = spark.read.csv('../data/raw/customers.csv', header=True, inferSchema=True)
spark_products = spark.read.csv('../data/raw/products.csv', header=True, inferSchema=True)
spark_orders = spark.read.csv('../data/raw/orders.csv', header=True, inferSchema=True)
spark_order_items = spark.read.csv('../data/raw/order_items.csv', header=True, inferSchema=True)

print("Data loaded successfully!")
print(f"Customers: {len(customers_df)}")
print(f"Products: {len(products_df)}")
print(f"Orders: {len(orders_df)}")
print(f"Order Items: {len(order_items_df)}")

## Customer Analysis

In [ ]:
# Customer demographics analysis
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Age distribution
axes[0, 0].hist(customers_df['age'], bins=20, alpha=0.7, color='skyblue')
axes[0, 0].set_title('Customer Age Distribution')
axes[0, 0].set_xlabel('Age')
axes[0, 0].set_ylabel('Frequency')

# Gender distribution
gender_counts = customers_df['gender'].value_counts()
axes[0, 1].pie(gender_counts.values, labels=gender_counts.index, autopct='%1.1f%%')
axes[0, 1].set_title('Customer Gender Distribution')

# State distribution (top 10)
state_counts = customers_df['state'].value_counts().head(10)
axes[1, 0].barh(state_counts.index, state_counts.values, color='lightcoral')
axes[1, 0].set_title('Top 10 States by Customer Count')
axes[1, 0].set_xlabel('Number of Customers')

# Registration trend
customers_df['registration_date'] = pd.to_datetime(customers_df['registration_date'])
monthly_registrations = customers_df.set_index('registration_date').resample('M').size()
axes[1, 1].plot(monthly_registrations.index, monthly_registrations.values, marker='o')
axes[1, 1].set_title('Customer Registrations Over Time')
axes[1, 1].set_xlabel('Date')
axes[1, 1].set_ylabel('Number of Registrations')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Product Analysis

In [ ]:
# Product analysis
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Category distribution
category_counts = products_df['category'].value_counts()
axes[0, 0].bar(category_counts.index, category_counts.values, color='lightgreen')
axes[0, 0].set_title('Products by Category')
axes[0, 0].set_xlabel('Category')
axes[0, 0].set_ylabel('Number of Products')
axes[0, 0].tick_params(axis='x', rotation=45)

# Price distribution
axes[0, 1].hist(products_df['price'], bins=30, alpha=0.7, color='gold')
axes[0, 1].set_title('Product Price Distribution')
axes[0, 1].set_xlabel('Price ($)')
axes[0, 1].set_ylabel('Frequency')

# Stock status
stock_counts = products_df['in_stock'].value_counts()
axes[1, 0].pie(stock_counts.values, labels=['In Stock', 'Out of Stock'], autopct='%1.1f%%')
axes[1, 0].set_title('Product Stock Status')

# Average price by category
avg_price_by_category = products_df.groupby('category')['price'].mean().sort_values(ascending=False)
axes[1, 1].barh(avg_price_by_category.index, avg_price_by_category.values, color='lightblue')
axes[1, 1].set_title('Average Price by Category')
axes[1, 1].set_xlabel('Average Price ($)')

plt.tight_layout()
plt.show()

## Order and Revenue Analysis

In [ ]:
# Merge orders with order items for revenue analysis
orders_with_items = orders_df.merge(order_items_df, on='order_id')
orders_with_items['total_price'] = orders_with_items['quantity'] * orders_with_items['unit_price']

# Convert dates
orders_with_items['order_date'] = pd.to_datetime(orders_with_items['order_date'])

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Order status distribution
status_counts = orders_df['order_status'].value_counts()
axes[0, 0].bar(status_counts.index, status_counts.values, color='plum')
axes[0, 0].set_title('Order Status Distribution')
axes[0, 0].set_xlabel('Status')
axes[0, 0].set_ylabel('Number of Orders')
axes[0, 0].tick_params(axis='x', rotation=45)

# Daily revenue trend
daily_revenue = orders_with_items.set_index('order_date').resample('D')['total_price'].sum()
axes[0, 1].plot(daily_revenue.index, daily_revenue.values, alpha=0.7)
axes[0, 1].set_title('Daily Revenue Trend')
axes[0, 1].set_xlabel('Date')
axes[0, 1].set_ylabel('Revenue ($)')
axes[0, 1].tick_params(axis='x', rotation=45)

# Payment method distribution
payment_counts = orders_df['payment_method'].value_counts()
axes[1, 0].pie(payment_counts.values, labels=payment_counts.index, autopct='%1.1f%%')
axes[1, 0].set_title('Payment Methods')

# Order value distribution
order_values = orders_with_items.groupby('order_id')['total_price'].sum()
axes[1, 1].hist(order_values, bins=50, alpha=0.7, color='lightcoral')
axes[1, 1].set_title('Order Value Distribution')
axes[1, 1].set_xlabel('Order Value ($)')
axes[1, 1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## Advanced PySpark Analysis

In [ ]:
# Use PySpark for advanced analysis
from pyspark.sql.functions import col, count, sum, avg, desc

# Join data in Spark
spark_orders_items = spark_orders.join(spark_order_items, "order_id")
spark_orders_items = spark_orders_items.withColumn("total_price", col("quantity") * col("unit_price"))

# Top 10 customers by spending
top_customers = spark_orders_items.join(spark_customers, "customer_id") \
    .groupBy("customer_id", "first_name", "last_name") \
    .agg(sum("total_price").alias("total_spent")) \
    .orderBy(desc("total_spent")) \
    .limit(10)

print("Top 10 Customers by Total Spending:")
top_customers.show()

In [ ]:
# Top 10 products by revenue
top_products = spark_orders_items.join(spark_products, "product_id") \
    .groupBy("product_id", "product_name", "category") \
    .agg(sum("total_price").alias("total_revenue"), \
         sum("quantity").alias("total_sold")) \
    .orderBy(desc("total_revenue")) \
    .limit(10)

print("Top 10 Products by Revenue:")
top_products.show()

In [ ]:
# Revenue by category
revenue_by_category = spark_orders_items.join(spark_products, "product_id") \
    .groupBy("category") \
    .agg(sum("total_price").alias("total_revenue"), \
         count("order_id").alias("total_orders")) \
    .orderBy(desc("total_revenue"))

print("Revenue by Category:")
revenue_by_category.show()

# Convert to pandas for visualization
category_revenue_pd = revenue_by_category.toPandas()

plt.figure(figsize=(12, 6))
plt.bar(category_revenue_pd['category'], category_revenue_pd['total_revenue'], color='skyblue')
plt.title('Total Revenue by Category')
plt.xlabel('Category')
plt.ylabel('Total Revenue ($)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Key Insights Summary

In [ ]:
# Generate summary statistics
print("=== E-COMMERCE DATA INSIGHTS ===")
print(f"\nTotal Customers: {len(customers_df):,}")
print(f"Total Products: {len(products_df):,}")
print(f"Total Orders: {len(orders_df):,}")
print(f"Total Order Items: {len(order_items_df):,}")

total_revenue = orders_with_items['total_price'].sum()
avg_order_value = orders_with_items.groupby('order_id')['total_price'].sum().mean()
print(f"\nTotal Revenue: ${total_revenue:,.2f}")
print(f"Average Order Value: ${avg_order_value:,.2f}")

print(f"\nTop Customer State: {customers_df['state'].mode().iloc[0]}")
print(f"Top Product Category: {products_df['category'].mode().iloc[0]}")
print(f"Most Common Payment Method: {orders_df['payment_method'].mode().iloc[0]}")

# Close Spark session
spark.stop()
print("\nAnalysis completed!")